In [1]:
import numpy as np
from matplotlib import pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('./daily_csv.csv')
df = df.dropna()
y = df['Price'].values
x = np.arange(1, len(y), 1)
minn, maxx = y.min(), y.max()
y = (y - minn) / (maxx - minn)
sequence_length = 10
X, Y = [], []
for i in range(5900):
    l = []
    for j in range(i, i + sequence_length):
        l.append(y[j])
    X.append(l)
    Y.append(y[i + sequence_length])
X = np.array(X)
Y = np.array(Y)

In [3]:
x

array([   1,    2,    3, ..., 5949, 5950, 5951])

In [4]:
y

array([0.1589214 , 0.15777395, 0.14687321, ..., 0.08089501, 0.07171543,
       0.06712565])

In [5]:
xtrain, xtest, ytrain, ytest = train_test_split(X, Y, test_size=0.10, random_state=42, shuffle= False, stratify= None)

In [6]:
class Data(Dataset):
    def __init__(self, x, y):
        super().__init__()
        self.x = torch.tensor(x, dtype= torch.float32)
        self.y = torch.tensor(y, dtype= torch.float32)
    def __getitem__(self, ind):
        return self.x[ind], self.y[ind]
    def __len__(self, ):
        return self.x.shape[0]

In [ ]:
class RNNModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = torch.nn.RNN(input_size=1, hidden_size= 5, num_layers= 1, batch_first= True)
        self.fc1 = torch.nn.Linear(in_features= 5, out_features= 1)
    def forward(self, x):
        y, h = self.rnn(x)
        y = y[: , -1, : ]
        y = self.fc1(torch.relu(y))
        return y

In [8]:
train = Data(xtrain, ytrain)
train_loader = DataLoader(train, shuffle= True, batch_size= 256, drop_last= True)
test = Data(xtest, ytest)
test_loader = DataLoader(test, batch_size= 256, drop_last= True)

In [9]:
model = RNNModel()
model = model.to('cuda')
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)

In [10]:
num_epochs = 1500

In [11]:
for epoch in range(num_epochs):
    running_loss = 0.0
    for input, target in train_loader:
        optimizer.zero_grad()
        input, target = input.reshape((-1, sequence_length, 1)).to('cuda'), target.to('cuda')
        output = model(input).reshape(-1)
        loss = criterion(output, target)
        loss.backward()
        running_loss += loss.item()
        optimizer.step()
    if epoch % 50 == 0:
        print(f'epoch - {epoch}, loss - {running_loss}')

epoch - 0, loss - 1.372048296034336
epoch - 50, loss - 0.2514777537435293
epoch - 100, loss - 0.1657884088344872
epoch - 150, loss - 0.07896991563029587
epoch - 200, loss - 0.028714939602650702
epoch - 250, loss - 0.017171088751638308
epoch - 300, loss - 0.011889734567375854
epoch - 350, loss - 0.009218006001901813
epoch - 400, loss - 0.007896568946307525
epoch - 450, loss - 0.007516333076637238
epoch - 500, loss - 0.007092392595950514
epoch - 550, loss - 0.006789645136450417
epoch - 600, loss - 0.006367906178638805
epoch - 650, loss - 0.006169009633595124
epoch - 700, loss - 0.005978398854495026
epoch - 750, loss - 0.005802346742711961
epoch - 800, loss - 0.0057041418040171266
epoch - 850, loss - 0.005597285387921147
epoch - 900, loss - 0.0054904166754568
epoch - 950, loss - 0.00540448918036418
epoch - 1000, loss - 0.00530249165603891
epoch - 1050, loss - 0.005273729453620035
epoch - 1100, loss - 0.005257742253888864
epoch - 1150, loss - 0.0051818195497617126
epoch - 1200, loss - 0.00

In [12]:
model.eval()
all_preds, all_labels = [], []
for input, target in test_loader:
    input, target = input.reshape((-1, sequence_length, 1)).to('cuda'), target.to('cuda')
    output = model(input).reshape(-1)
    all_preds.extend(output.to('cpu').detach().numpy())
    all_labels.extend(target.to('cpu').detach().numpy())

In [13]:
from sklearn.metrics import mean_squared_error
print(mean_squared_error(all_preds, all_labels))

4.705983902075767e-05
